# Conway's Game of Life

Conway's **Game of Life** (1970) is a two-dimensional cellular automaton invented by mathematician John Horton Conway. It is zero-player: given an initial configuration, the entire future evolution is deterministic. Despite its four simple rules, it supports emergent behaviors of extraordinary complexity — gliders, oscillators, guns, and even Turing-complete computation.

The Game of Life is a foundational example in the study of **complex systems**: how local, simple interactions can give rise to global, complicated structure.

## The Rules

The state space is an (infinite) two-dimensional grid. Each cell $(i, j)$ is either **alive** ($= 1$) or **dead** ($= 0$). Let

$$N(i,j) = \sum_{(di,\,dj) \in \{-1,0,1\}^2 \setminus \{(0,0)\}} c_{i+di,\,j+dj}$$

denote the number of live cells in the **Moore neighborhood** (the 8 surrounding cells). The update rules are:

| Condition | Outcome |
|-----------|--------|
| Live cell, $N < 2$ | Dies (underpopulation) |
| Live cell, $N \in \{2, 3\}$ | Survives |
| Live cell, $N > 3$ | Dies (overpopulation) |
| Dead cell, $N = 3$ | Born (reproduction) |

Compactly: **B3/S23** (born with 3 neighbors, survives with 2 or 3).

All cells update **simultaneously** — each new generation is computed from the current generation, not from partially updated states.

## Efficient Neighbor Counting

Rather than looping over every cell and its 8 neighbors in Python, we sum 8 **shifted copies** of the grid using `np.roll`. Shifting the grid by $(\Delta i, \Delta j)$ brings each cell's neighbor in that direction into alignment, so summing all 8 shifts gives the neighbor count for every cell at once.

`np.roll` wraps at the boundaries, implementing **toroidal (periodic) boundary conditions** — cells that leave one edge reappear on the opposite edge.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

plt.style.use("dark_background")

In [2]:
def count_neighbors(grid):
    """Sum of the 8 Moore neighbors at every cell (vectorized via np.roll)."""
    N = np.zeros_like(grid, dtype=np.int32)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            N += np.roll(np.roll(grid, di, axis=0), dj, axis=1)
    return N


def gol_step(grid):
    """Apply one generation of the B3/S23 rules."""
    N = count_neighbors(grid)
    birth   = (grid == 0) & (N == 3)
    survive = (grid == 1) & ((N == 2) | (N == 3))
    return (birth | survive).astype(np.uint8)


def precompute_frames(grid, n_steps):
    """Run the automaton for n_steps generations and return all frames
    as a single array of shape (n_steps+1, rows, cols)."""
    rows, cols = grid.shape
    frames = np.empty((n_steps + 1, rows, cols), dtype=np.uint8)
    frames[0] = grid
    for t in range(1, n_steps + 1):
        frames[t] = gol_step(frames[t - 1])
    return frames

## Initial Conditions

Four canonical initial conditions produce qualitatively different behaviors:

| Pattern | Wolfram class | What happens |
|---------|--------------|-------------|
| **Blinker** | II — periodic | Single oscillator, period 2, toggles horizontal ↔ vertical forever |
| **Glider** | IV — complex | 5-cell spaceship translates diagonally 1 cell every 4 generations |
| **Gosper gun** | IV — complex | Emits one new glider every 30 generations; population grows without bound |
| **Random** | Mix | A random soup self-organizes into still lifes, oscillators, and the occasional glider |

In [3]:
def make_blinker(rows=30, cols=30):
    """Horizontal blinker at the center of a rows x cols grid."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    r, c = rows // 2, cols // 2
    g[r, c-1:c+2] = 1
    return g


def make_glider(rows=50, cols=50):
    """Classic glider near the top-left corner of a rows x cols grid."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    for r, c in [(0,1),(1,2),(2,0),(2,1),(2,2)]:
        g[r+2, c+2] = 1
    return g


def make_gosper_gun(rows=50, cols=100):
    """Gosper glider gun — fires one glider every 30 generations."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    cells = [
        (5,1),(5,2),(6,1),(6,2),
        (5,11),(6,11),(7,11),(4,12),(8,12),(3,13),(3,14),(9,13),(9,14),
        (6,15),(4,16),(8,16),(5,17),(6,17),(7,17),(6,18),
        (3,21),(4,21),(5,21),(3,22),(4,22),(5,22),(2,23),(6,23),
        (1,25),(2,25),(6,25),(7,25),
        (3,35),(4,35),(3,36),(4,36),
    ]
    for r, c in cells:
        if 0 <= r < rows and 0 <= c < cols:
            g[r, c] = 1
    return g


def make_random(rows=70, cols=70, density=0.30, seed=42):
    """Random soup with the given fraction of live cells."""
    rng = np.random.default_rng(seed)
    return (rng.random((rows, cols)) < density).astype(np.uint8)

## Animation

Set `PATTERN` to one of `'blinker'`, `'glider'`, `'gun'`, or `'random'`, then run the cell. All frames are precomputed before the animation starts so playback is smooth.

The generation counter and live-cell count update each frame. Boundaries are toroidal.

In [6]:
# ── Change this to: 'blinker' | 'glider' | 'gun' | 'random' ─────────────────
PATTERN = 'gun'

# --- Global animation speed (ms per frame) ---
FRAME_MS = 150

# ────────────────────────────────────────────────────────────────────────────

configs = {
    'blinker': dict(
        grid    = make_blinker(rows=30, cols=30),
        n_steps = 20,
        title   = 'Blinker — Period-2 Oscillator',
        figsize = (5, 5),
    ),
    'glider': dict(
        grid    = make_glider(rows=50, cols=50),
        n_steps = 120,
        title   = 'Glider — Period-4 Spaceship',
        figsize = (5, 5),
    ),
    'gun': dict(
        grid    = make_gosper_gun(rows=50, cols=100),
        n_steps = 210,
        title   = 'Gosper Glider Gun',
        figsize = (10, 5),
    ),
    'random': dict(
        grid    = make_random(rows=70, cols=70, density=0.30, seed=42),
        n_steps = 200,
        title   = 'Random Soup (density = 0.30)',
        figsize = (5, 5),
    ),
}

cfg    = configs[PATTERN]
frames = precompute_frames(cfg['grid'], cfg['n_steps'])

fig, ax = plt.subplots(figsize=cfg['figsize'])
ax.axis('off')

# --- Equal aspect ratio ---
ax.set_aspect('equal')

img = ax.imshow(
    frames[0],
    cmap='binary_r',
    interpolation='nearest',
    vmin=0,
    vmax=1
)

title_txt = ax.set_title(
    f"{cfg['title']}   gen 0   live: {int(frames[0].sum())}",
    fontsize=11,
    pad=6
)

def update(t):
    img.set_data(frames[t])
    title_txt.set_text(
        f"{cfg['title']}   gen {t}   live: {int(frames[t].sum())}"
    )
    return [img, title_txt]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=FRAME_MS,   # ← single constant now
    blit=False
)

plt.tight_layout()
plt.close(fig)

HTML(anim.to_jshtml())

## Boundary Conditions and Wolfram Classification

All four initial conditions use **toroidal (periodic) boundaries**: cells exiting one edge reappear on the opposite side. This removes edge effects and lets the blinker oscillate and the glider travel indefinitely. For the gun, gliders that cross the right edge wrap back to the left; on the 50×100 grid they return after several hundred generations.

**Wolfram's classification** (1984) organizes cellular automata into four classes:

| Class | Behavior | Example here |
|-------|----------|--------------|
| I | All cells die or freeze | Dense random seeds at extreme density |
| II | Stable oscillators and still lifes | Blinker |
| III | Chaotic, aperiodic | High-density random soup |
| IV | Complex localized structures | Glider, gun, sparse random soup |

GoL itself is Class IV — it can support all classes of behavior in different spatial regions simultaneously. The random seed typically produces a Class III transient that settles into a Class II mix of still lifes and oscillators, occasionally emitting Class IV gliders.